# accuracy_review 점검

합성 청크와 추출 결과를 준비하고, 판정표를 만든 뒤 집계합니다. 실제 카드 원문은 사용하지 않습니다.

In [ ]:
from pathlib import Path
import json
import tempfile

work_dir = Path(tempfile.mkdtemp(prefix="accuracy-review-check-"))
result_dir = work_dir / "results"
chunk_dir = work_dir / "chunks"
output_dir = work_dir / "review"
result_dir.mkdir()
chunk_dir.mkdir()

## 합성 입력과 모듈 호출

In [ ]:
chunk_id = "synthetic-001"
chunk = {"chunk_id": chunk_id, "page_no": 1, "text": "합성 안내문: 지난달 40만원 이상 이용 시 5% 할인 예시입니다."}
result = {"chunk_id": chunk_id, "page_no": 1, "chunk_length": len(chunk["text"]),
          "extraction": {"benefits": [{"name": "합성 혜택", "rate": "5%", "method": "할인",
                                           "evidence": [{"text": "합성 안내문: 지난달 40만원 이상 이용 시 5% 할인 예시입니다."}]}],
                         "conditions": [{"kind": "SPEND_REQ", "text": "합성 조건", "min_amount": 400000,
                                          "evidence": [{"text": "합성 안내문: 지난달 40만원 이상 이용 시 5% 할인 예시입니다."}]}],
                         "limits": [], "targets": [], "relations": []}}
(chunk_dir / "synthetic.chunks.json").write_text(json.dumps({"chunks": [chunk]}, ensure_ascii=False), encoding="utf-8")
(result_dir / "synthetic.result.json").write_text(json.dumps(result, ensure_ascii=False), encoding="utf-8")
from payment_card.file_pipeline.extraction.review.accuracy_review import auto_check, prepare, tally
synthetic_checks = auto_check(result["extraction"], chunk["text"])
synthetic_checks  # 합성 숫자 대조와 숫자 확인 힌트
prepared = prepare(result_dir, chunk_dir, output_dir, size=1, seed=7)
prepared["selected"], prepared["sheet"]

## 판정 집계

In [ ]:
sheet_path = Path(prepared["sheet"])
sheet_text = sheet_path.read_text(encoding="utf-8")
sheet_path.write_text(sheet_text.replace("[ ] 맞음", "[x] 맞음", 1), encoding="utf-8")
tally(sheet_path)["report"]